# acomplete

> A high level unified function make api calls

In [ ]:
#| default_exp acomplete

## Imports

In [ ]:
#| export
import asyncio,json,httpx
from importlib.resources import files
from fastcore.utils import *
from fastcore.meta import *
from fastspec.spec import *
from fastspec.oapi import *
from fastspec.errors import APIError

from fastllm.types import *
from fastllm.streaming import *
from fastllm.openai_responses import *
from fastllm.openai_chat import *
from fastllm.anthropic import *
from fastllm.gemini import *

In [ ]:
#| hide
import nbdev,base64,httpx,socket
from fastcore.test import *
from toolslm.funccall import get_schema
from cachy.core import enable_cachy, disable_cachy, doms

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
#| export
specs_path = files('fastllm') / 'specs'
ant_spec  = SpecParser.from_openapi(dict2obj(json.loads((specs_path/'anthropic.json').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(json.loads((specs_path/'openai.with-code-samples.json').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

## Infer Client

Utils to infer client setup from model name. Users can just pass model name to `acomplete` and the provider will be inferred and client will be create automatically.

A vendor consists of a registered api name, a base url, and a default api key env var name

In [ ]:
#| export
_codex_path = os.getenv('CODEX_AUTH_PATH', '~/.codex/auth.json')
_codex_json = _codex_path, ('tokens','access_token')
vendor_mapping = {
    "openai":       ('openai', 'https://api.openai.com/v1', 'OPENAI_API_KEY'),
    "anthropic":    ('anthropic', 'https://api.anthropic.com', 'ANTHROPIC_API_KEY'),
    "gemini":       ('gemini', 'https://generativelanguage.googleapis.com/', 'GEMINI_API_KEY'),
    "openai_chat":  ('openai_chat', 'https://api.openai.com/v1', 'OPENAI_API_KEY'),
    "codex":        ('openai', 'https://chatgpt.com/backend-api/codex', 'CODEX_AUTH_TOKEN', _codex_json),
    "moonshot":     ('openai_chat', "https://api.moonshot.ai/v1", "MOONSHOT_API_KEY"),
    "deepseek":     ('openai_chat', "https://api.deepseek.com/v1", "DEEPSEEK_API_KEY"),
    "mimo":         ('openai_chat', "https://api.xiaomimimo.com/v1", "MIMO_API_KEY"),
    "openrouter":   ('openai_chat', "https://openrouter.ai/api/v1", "OPENROUTER_API_KEY"),
    "together":     ('openai_chat', "https://api.together.xyz/v1", "TOGETHER_API_KEY"),
    "fireworks_ai": ('openai_chat', "https://api.fireworks.ai/inference/v1", "FIREWORKS_API_KEY"),
    "qwen":         ('openai_chat', "https://dashscope.aliyuncs.com/compatible-mode/v1", "QWEN_API_KEY"),
    "minimax":      ('anthropic', "https://api.minimax.io/anthropic", "MINIMAX_API_KEY")
}

A registered api name to OpenAPIClient spec mapping:

In [ ]:
#| export
api2spec = {'openai':oai_spec, 'openai_chat':oai_spec, 'anthropic':ant_spec, 'gemini':gem_spec}

A utility to auto resolve api for openai (uses responses api), anthropic, and gemini models:

In [ ]:
api_registry.apis.keys()

dict_keys(['openai', 'openai_chat', 'anthropic', 'gemini'])

In [ ]:
#| export
@flexicache()
def mk_client(model=None, vendor_name=None, api_name=None, api_key=None, base_url=None, xtra_hdrs=None,
    timeout=httpx.Timeout(connect=30, read=300, write=30, pool=10)):
    err_msg = f"please pass a valid one vendor: {', '.join(list(vendor_mapping))} or pass `api_name`,`base_url` and `api_key`"
    if vendor_name:
        override_base_url = base_url
        try:
            api_name, base_url, env_api_nm, *auth_json = vendor_mapping[vendor_name]
            base_url = override_base_url or base_url
            if auth_json and not api_key and not os.getenv(env_api_nm):
                fn,keys = auth_json[0]  # pyright: ignore[reportAssignmentType]
                auth_fn = Path(fn).expanduser()
                if auth_fn.exists(): api_key = nested_idx(json.loads(auth_fn.read_text()), *keys)
            api_key = get_api_key(api_key, env_api_nm)
        except KeyError: raise ValueError(f"Unknown vendor '{vendor_name}', {err_msg}")
    elif base_url and api_key: vendor_name, api_name = ifnone(vendor_name, 'custom'), ifnone(api_name, 'openai_chat')
    elif (api_name:=infer_api_name(model)):  base_url, vendor_name = None, api_name
    else: raise ValueError(f"Model {model} can't be auto resolved, {err_msg}")
    api = api_registry.apis[api_name]
    spec, hdrs = api2spec[api_name], api.get_hdrs(api_key)
    cli = OpenAPIClient(spec, headers=merge(hdrs, ifnone(xtra_hdrs, {})), timeout=timeout)
    if base_url is not None:
        for op in cli.ops: op.base_url = base_url  # pyright: ignore[reportAttributeAccessIssue]
    return cli, api_name, vendor_name

#### Auto Resolved

In [ ]:
cli, api_name, vendor_name = mk_client('models/gemini-model')

In [ ]:
test_eq(cli.ops[0].base_url, 'https://generativelanguage.googleapis.com/')
test_eq(api_name, 'gemini')
test_eq(vendor_name, 'gemini')

OpenAI gpt models default to Responses API:

In [ ]:
cli, api_name, vendor_name = mk_client('gpt-model')

In [ ]:
test_eq(cli.ops[0].base_url, 'https://api.openai.com/v1')
test_eq(api_name, 'openai')
test_eq(vendor_name, 'openai')

In [ ]:
cli, api_name, vendor_name = mk_client('claude-model')

In [ ]:
test_eq(cli.ops[0].base_url, 'https://api.anthropic.com')
test_eq(api_name, 'anthropic')
test_eq(vendor_name, 'anthropic')

#### Known Vendor

Users can also pass a model name with a known vendor, to force openai-chat, to use codex, or other 3rd party providers that support an openai chat endpoint:

In [ ]:
cli, api_name, vendor_name = mk_client('gpt-model', 'openai_chat') # Using a gpt model with openai chat api

In [ ]:
test_eq(cli.ops[0].base_url, 'https://api.openai.com/v1')
test_eq(api_name, 'openai_chat')
test_eq(vendor_name, 'openai_chat')

In [ ]:
os.environ['CODEX_AUTH_TOKEN'] = 'my_codex_token'

In [ ]:
cli, api_name, vendor_name = mk_client('gpt-codex', 'codex') # Using a gpt model with openai chat api

In [ ]:
test_eq(cli.ops[0].base_url, 'https://chatgpt.com/backend-api/codex')
test_eq(api_name, 'openai') # uses responses api
test_eq(vendor_name, 'codex')

In [ ]:
cli, api_name, vendor_name = mk_client('kimi-2.5', 'fireworks_ai') # Using a gpt model with openai chat api

In [ ]:
test_eq(cli.ops[0].base_url, 'https://api.fireworks.ai/inference/v1')
test_eq(api_name, 'openai_chat')
test_eq(vendor_name, 'fireworks_ai')

In [ ]:
try: cli, api_name, vendor_name = mk_client('kimi-2.5', 'unknown')
except Exception as e: print(e)

Unknown vendor 'unknown', please pass a valid one vendor: openai, anthropic, gemini, openai_chat, codex, moonshot, deepseek, mimo, openrouter, together, fireworks_ai, qwen, minimax or pass `api_name`,`base_url` and `api_key`


In [ ]:
try: cli, api_name, vendor_name = mk_client('unknown model')
except Exception as e: print(e)

Model unknown model can't be auto resolved, please pass a valid one vendor: openai, anthropic, gemini, openai_chat, codex, moonshot, deepseek, mimo, openrouter, together, fireworks_ai, qwen, minimax or pass `api_name`,`base_url` and `api_key`


#### Custom

Finally users can pass `base_url` and `api_key` to use a custom openai chat endpoint:

In [ ]:
os.environ['CUSTOM_API_KEY'] = 'my_key'

In [ ]:
cli, api_name, vendor_name = mk_client('my-vllm-model', api_name='openai_chat', base_url='https://api.custom.com', api_key=os.getenv('CUSTOM_API_KEY'))

In [ ]:
test_eq(cli.ops[0].base_url, 'https://api.custom.com')
test_eq(api_name, 'openai_chat')
test_eq(vendor_name, 'custom')

## Acomplete

### Error Classification

Different providers signal "context window exceeded" in different ways — OpenAI uses `code: "context_length_exceeded"`, while Anthropic and Gemini embed hints in the error message (e.g. "prompt is too long", "exceeds the maximum number of tokens allowed"). We classify these into a dedicated `ContextWindowExceededError` subclass so callers (e.g. lisette's tool loop) can catch and recover specifically.

- **OpenAI** (Chat & Responses): HTTP 400 with `error.type = "invalid_request_error"` and `error.code = "context_length_exceeded"`. Message starts with "This model's maximum context length is …". See [OpenAI docs troubleshooting](https://devidevs.com/blog/openai-api-errors-troubleshooting).
- **Anthropic**: HTTP 400 with `error.type = "invalid_request_error"` and message `"prompt is too long: N tokens > M maximum"`. See [continuedev/continue#6270](https://github.com/continuedev/continue/issues/6270).
- **Gemini**: HTTP 400 with `error.status = "INVALID_ARGUMENT"` and message `"The input token count (N) exceeds the maximum number of tokens allowed (M)."`. See [gemini-cli#11939](https://github.com/google-gemini/gemini-cli/issues/11939).

Substring list here is derived from [litellm's `is_error_str_context_window_exceeded`](https://github.com/BerriAI/litellm/blob/main/litellm/litellm_core_utils/exception_mapping_utils.py).

In [ ]:
#| export
class ContextWindowExceededError(APIError): pass

def _is_ctx_exceeded(code, msg):
    m = (msg or "").lower()
    if any(x in m for x in ("string_above_max_length", "invalid 'user'")): return False
    if str(code or "").lower() == "context_length_exceeded": return True
    return any(s in m for s in ("exceed context limit", "maximum context length", "maximum context limit",
    "longer than the model's context length", "input tokens exceed the configured limit",
    "exceeds the maximum number of tokens allowed", "prompt is too long", "exceeds the context window"))

def _classify_error(exc):
    "Upgrade generic `APIError` to a specific subclass if applicable."
    if not isinstance(exc, APIError): return exc
    if _is_ctx_exceeded(exc.code, exc.message):
        return ContextWindowExceededError(exc.message, provider=exc.provider, model=exc.model,
            endpoint=exc.endpoint, status_code=exc.status_code, error_type=exc.error_type,
            code=exc.code, request_id=exc.request_id, retryable=exc.retryable, raw=exc.raw)
    return exc

async def _classify_error_stream(gen):
    "Wrap an async generator to upgrade `APIError`s as they're raised during iteration."
    try:
        async for x in gen: yield x
    except APIError as e: raise _classify_error(e) from e

In [ ]:
cli, api_name, vendor_name = mk_client('deepseek-v4-flash', base_url='https://api.deepseek.com/v1', api_key=os.getenv('DEEPSEEK_API_KEY'))

In [ ]:
cli, api_name, vendor_name

(<fastspec.oapi.OpenAPIClient>, 'openai_chat', 'custom')

In [ ]:
e_oai = APIError("This model's maximum context length is 4097 tokens...", status_code=400,
                 error_type="invalid_request_error", code="context_length_exceeded")
e_ant = APIError("prompt is too long: 210503 tokens > 200000 maximum", status_code=400,
                 error_type="invalid_request_error")
e_gem = APIError("The input token count (1632254) exceeds the maximum number of tokens allowed (1048576).",
                 status_code=400, error_type="INVALID_ARGUMENT")

test(_classify_error(e_oai), ContextWindowExceededError, isinstance)
test(_classify_error(e_ant), ContextWindowExceededError, isinstance)
test(_classify_error(e_gem), ContextWindowExceededError, isinstance)

### acomplete()

In [ ]:
#| export
defaults = SimpleNamespace(debug_mode=False)

def _debug_print(model, api_name, vendor_name, payload, func):
    "Pretty-print acomplete inputs when defaults.debug_mode is set"
    from pprint import pformat
    p = dict(payload)
    if defaults.debug_mode == 'brief' and 'tools' in p:
        p['tools'] = '; '.join(o.get('name', o.get('type', o)) for o in p['tools'])
    print('━'*60)
    print(f"\033[1;36mfastllm debug\033[0m  model={model} vendor={vendor_name} api={api_name} base_url={func.base_url} path={func.path}")
    print('─'*60)
    print(f"\033[1;33mpayload:\033[0m\n{pformat(p, width=120, sort_dicts=False)}")
    print('━'*60)

### Retries

A request can be retried while no output has reached the caller. After a stream has yielded a chunk, retrying would mix two different completions, so the error is raised instead.

In [ ]:
#| export
async def _raise_if_done(e, n, retries, retry_delay, yielded=False):
    e = _classify_error(e)
    if yielded or not e.retryable or n == retries: raise e
    await asyncio.sleep(retry_delay*2**n)

async def _retry_call(f, retries=2, retry_delay=0.5):
    for n in range(retries+1):
        try: return await f()
        except APIError as e: await _raise_if_done(e, n, retries, retry_delay)

async def _retry_stream(mk_gen, retries=2, retry_delay=0.5):
    for n in range(retries+1):
        yielded = False
        try:
            async for o in mk_gen():
                yielded = True
                yield o
            return
        except APIError as e: await _raise_if_done(e, n, retries, retry_delay, yielded=yielded)

In [ ]:
#| export
@delegates(payload_kwargs)
async def acomplete(msgs, model, api_name=None, vendor_name=None, api_key=None,
                    base_url=None, xtra_body=None, xtra_hdrs=None, stream=False,
                    stop_callables=None, retries=2, retry_delay=0.5, **kwargs):
    "Unified completion across different APIs."
    cli, api_name, vendor_name = mk_client(model, vendor_name, api_name, api_key, base_url, xtra_hdrs)
    api = api_registry.apis[api_name]
    payload = api.mk_payload(msgs, model, stream=stream, **kwargs)
    payload = merge(payload, ifnone(xtra_body, {}))
    if vendor_name == 'codex':
        for k in 'temperature max_tokens max_output_tokens max_completion_tokens metadata'.split(): payload.pop(k, None)
        payload['store'] = False
    if nested_idx(payload, 'messages', -1, 'role') == 'assistant':
        if vendor_name == 'deepseek' and 'v4' in model:   payload['messages'][-1]['prefix'] = True
        if vendor_name == 'moonshot' and 'kimi' in model: payload['messages'][-1]['partial'] = True
    func = attrgetter(api.op_path[stream])(cli)
    if defaults.debug_mode: _debug_print(model, api_name, vendor_name, payload, func)
    async def _call(): return await func(**payload)
    if not stream:
        resp = await _retry_call(_call, retries, retry_delay)
        return mk_completion(resp, model=model, api_name=api_name, vendor_name=vendor_name)

    async def _mk_gen():
        resp = await _call()
        async for o in api.acollect_stream(resp, model=model, vendor_name=vendor_name, stop_callables=stop_callables): yield o

    return _retry_stream(_mk_gen, retries, retry_delay)

In [ ]:
@delegates(acomplete)
async def print_stream(msgs, model, max_print=100, **kwargs):
    cnt,printed,done = 0,0,False
    async for o in await acomplete(msgs, model, stream=True, **kwargs): 
        if not isinstance(o, (Completion,Part)) and not done: 
            if o.get('thinking') and cnt<10: print('🧠',end='',flush=True)
            if (txt:=o.get('text')):
                if max_print is not None and printed+len(txt)>max_print:
                    txt,done = txt[:max_print-printed]+'…',True
                print(txt,end='',flush=True)
                printed += len(txt)
            cnt+=1
    return o

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Hello!')])

In [ ]:
comp = await print_stream([msg1], model='gpt-4o-mini'); comp

Hello

!

 How

 can

 I

 assist

 you

 today

?

<div class="prose" markdown="1">

Hello! How can I assist you today?

<details markdown='1'>

- model: `gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 9, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 10, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 19})`

</details>

</div>

In [ ]:
comp = await print_stream([msg1], model='MiniMax-M3', vendor_name='minimax'); comp

Hello

! How can I help you today?

<div class="prose" markdown="1">

Hello! How can I help you today?

<details markdown='1'>

- model: `MiniMax-M3`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=165, completion_tokens=9, total_tokens=174, cached_tokens=114, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 51, 'output_tokens': 9, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 114})`

</details>

</div>

### Retry tests

These small fakes let us check three cases: failure before the stream starts, failure before the first chunk, and failure after a chunk has already been yielded.

In [ ]:
class FakeCli:
    def __init__(self, fail_n=0): self.n,self.fail_n = 0,fail_n
    async def stream(self, **kw):
        self.n += 1
        if self.n <= self.fail_n: raise APIError("upstream fail", status_code=503, retryable=True)
        return 'resp'

class FakeApi:
    op_path = {True:'stream', False:'complete'}
    def __init__(self, fail_n=0, late=False): self.n,self.fail_n,self.late = 0,fail_n,late
    def mk_payload(self, *a, **kw): return {}
    async def acollect_stream(self, resp, **kw):
        self.n += 1
        if self.late: yield dict(text='bad')
        if self.n <= self.fail_n: raise APIError("stream fail", status_code=503, retryable=True)
        yield dict(text='ok')

In [ ]:
async def _txts(rs):
    txt = ''
    async for o in rs:
        if not isinstance(o, Completion): txt += o.get('text', '')
    return txt

async def run_retry(cli, api):
    global mk_client
    old = mk_client
    api_registry.apis['retry_test'] = api
    try:
        mk_client = lambda *a, **kw: (cli, 'retry_test', 'retry_test')
        rs = await acomplete([msg1], 'm', stream=True, retry_delay=0)
        return await _txts(rs)
    finally:
        mk_client = old
        del api_registry.apis['retry_test']

In [ ]:
cli,api = FakeCli(fail_n=1),FakeApi()
test_eq(await run_retry(cli,api), 'ok')
test_eq((cli.n,api.n), (2,1))

In [ ]:
cli,api = FakeCli(),FakeApi(fail_n=1)
test_eq(await run_retry(cli,api), 'ok')
test_eq((cli.n,api.n), (2,2))

In [ ]:
cli,api = FakeCli(),FakeApi(fail_n=1,late=True)
with expect_fail(): await run_retry(cli,api)
test_eq((cli.n,api.n), (1,1))

The network tests check that connection failures and timeouts become `APIError`s. They use local ports so the tests do not depend on any provider being available.

In [ ]:
def hangport():
    s = socket.socket(); s.bind(('127.0.0.1', 0)); s.listen(1)
    return s, s.getsockname()[1]

srv,port = hangport()
try:
    base_url = f'http://127.0.0.1:{port}'
    cli,*_ = mk_client('gpt-4o-mini', None, None, 'x', base_url, None)
    for op in cli.ops: op.client.client._timeout = httpx.Timeout(0.5)
    with expect_fail(exc=APIError): await acomplete([msg1], 'gpt-4o-mini', base_url=base_url, api_key='x', retries=0)
finally: srv.close()

In [ ]:
def deadport():
    s = socket.socket(); s.bind(('127.0.0.1', 0)); port = s.getsockname()[1]; s.close()
    return port

port = deadport()
with expect_fail(exc=APIError): await acomplete([msg1], 'gpt-4o-mini', base_url=f'http://127.0.0.1:{port}', api_key='x', retries=0)

To use openai models with chat completions api we can set `vendor_name="openai_chat"`:

In [ ]:
comp = await print_stream([msg1], model='gpt-4o-mini', vendor_name='openai_chat'); comp

Hello

!

 How

 can

 I

 assist

 you

 today

?

<div class="prose" markdown="1">

Hello! How can I assist you today?

<details markdown='1'>

- model: `gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=9, total_tokens=18, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 9, 'completion_tokens': 9, 'total_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})`

</details>

</div>

In [ ]:
comp = await print_stream([msg1], model='claude-sonnet-4-20250514'); comp

Hello! It

's nice to meet you. How are you doing today? Is there anything I can help you with or woul…

<div class="prose" markdown="1">

Hello! It's nice to meet you. How are you doing today? Is there anything I can help you with or would you like to chat about?

<details markdown='1'>

- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=34, total_tokens=43, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 9, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 34})`

</details>

</div>

In [ ]:
comp = await print_stream([msg1], model='models/gemini-3-flash-preview'); comp

Hello! How can

 I help you today?

<div class="prose" markdown="1">

Hello! How can I help you today?

<details markdown='1'>

- model: `models/gemini-3-flash-preview`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=3, completion_tokens=9, total_tokens=190, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=178, raw={'promptTokenCount': 3, 'candidatesTokenCount': 9, 'totalTokenCount': 190, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3}], 'thoughtsTokenCount': 178, 'serviceTier': 'standard'})`

</details>

</div>

In [ ]:
comp = await print_stream([msg1], model='accounts/fireworks/models/kimi-k2p5', vendor_name='fireworks_ai'); comp

🧠

🧠

🧠

🧠

🧠

...

 help

 you

 today

?

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

The user just said "Hello!" which is a simple greeting. I should respond in a friendly, helpful manner. Since this is the beginning of the conversation, I should welcome them and ask how I can help them today.

I need to:
1. Greet them back warmly
2. Ask how I can assist them
3. Keep it concise but welcoming

No need for anything complicated here - just a standard friendly greeting response.

</details>

Hello! How can I help you today?

<details markdown='1'>

- model: `accounts/fireworks/models/kimi-k2p5`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=10, completion_tokens=99, total_tokens=109, cached_tokens=3, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 10, 'total_tokens': 109, 'completion_tokens': 99, 'prompt_tokens_details': {'cached_tokens': 3}})`

</details>

</div>

In [ ]:
comp = await print_stream([msg1], model='accounts/fireworks/models/kimi-k2p5', vendor_name='fireworks_ai', xtra_body={'service_tier':'priority'}); comp

🧠

🧠

🧠

🧠

🧠

...

 help

 you

 today

?

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

The user said "Hello!" which is a simple greeting. I should respond in a friendly, helpful manner. Since this is the start of the conversation, I should be welcoming and ask how I can help them today.

I don't need to overthink this - just a warm greeting back and an offer to assist with whatever they need.

</details>

Hello! How can I help you today?

<details markdown='1'>

- model: `accounts/fireworks/models/kimi-k2p5`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=10, completion_tokens=78, total_tokens=88, cached_tokens=9, cache_creation_tokens=0, reasoning_tokens=0, raw={'prompt_tokens': 10, 'total_tokens': 88, 'completion_tokens': 78, 'prompt_tokens_details': {'cached_tokens': 9}})`

</details>

</div>

### Tests

Model vendor pairs for testing:

In [ ]:
models = [('gpt-4o-mini',None), ('gpt-4o-mini', 'openai_chat'), ('claude-sonnet-4-6',None), ('models/gemini-3-flash-preview',None)]
think_models = [('o4-mini',None),('accounts/fireworks/models/kimi-k2p5', 'fireworks_ai'), ('claude-sonnet-4-6',None), ('models/gemini-3-flash-preview',None)]

#### Text

In [ ]:
def mk_user_msg(txt): return Msg(role='user', content=[Part(type=PartType.text, text=txt)])

In [ ]:
msg1 = mk_user_msg('Say hi in one word')
comps = {mn: await acomplete([msg1], model=mn, vendor_name=vnm) for mn,vnm in models}
for mn1,_ in models:
    for mn,vnm in models:
        msg2, msg3 = comps[mn1].message, mk_user_msg('Now say bye in one word')
        print(f'  {mn1:12} -> {mn:12}: ', end='')
        await print_stream([msg1, msg2, msg3], model=mn, vendor_name=vnm)
        print()

  gpt-4o-mini  -> gpt-4o-mini : Fare

well

!


  gpt-4o-mini  -> gpt-4o-mini : Good

well

...

.


  models/gemini-3-flash-preview -> claude-sonnet-4-6: Bye.

Bye


  models/gemini-3-flash-preview -> models/gemini-3-flash-preview: 

Bye

#### Thinking

Anthropic uses *adaptive* thinking (`denorm_anthropic_reasoning` sets `{"thinking": {"type": "adaptive"}, ...}`), and Gemini's `thinkingLevel='medium'` is also adaptive. On a trivial question like `789 * 2`, the model may legitimately decide to skip thinking. Kimi's via `reasoning_content` seems less adaptive (still streams thought blocks). o4-mini never shows thinking because OpenAI Responses doesn't emit reasoning text unless `summary: "auto"` is set (you had it in an earlier cell but not in `acomplete`).

In [ ]:
msg1 = mk_user_msg('What is 123 * 456?')
comps = {mn: await acomplete([msg1], model=mn, vendor_name=vnm, reasoning_effort='low') for mn,vnm in think_models}
comps2 = {}
for mn1,_ in think_models:
    for mn,vnm in think_models:
        msg2, msg3 = comps[mn1].message, mk_user_msg('What is 31231231 * 12312?')
        print(f'  {mn1:12} -> {mn:12}: ', end='')
        comps2[mn] = await print_stream([msg1, msg2, msg3], model=mn, vendor_name=vnm, reasoning_effort='medium')
        print()

  o4-mini      -> o4-mini     : 312

312

31

 ×

123

...

 value components:

| Component | Ca…


  models/gemini-3-flash-preview -> models/gemini-3-flash-preview: 🧠

🧠

31,231,231 * 12,31

2 = **384,518,916,072**

#### Tool Call

In [ ]:
tools = [sch]
msg1 = mk_user_msg('What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')
comps = {mn: await acomplete([msg1], model=mn, vendor_name=vnm, tools=tools) for mn,vnm in models}
for mn1,_ in models:
    for mn,vnm in models:
        msg2 = comps[mn1].message
        trmsg = mk_tool_res_msg(comps[mn1].tool_calls, ('8','30'))
        print(f'  {mn1:12} -> {mn:12}: ', end='')
        await print_stream([msg1, msg2, trmsg], model=mn, tools=tools, vendor_name=vnm)
        print()

  gpt-4o-mini  -> gpt-4o-mini : The


  gpt-4o-mini  -> gpt-4o-mini : The

 result

 of

 \(

...


  models/gemini-3-flash-preview -> claude-sonnet-4-6: Here are the results:

- **

3 + 5 = 8**
- **10 + 20 = 30**

Both calculations were performed in paral…


  models/gemini-3-flash-preview -> models/gemini-3-flash-preview: OK

. 3 + 5 is 8, and 10 + 20

 is 30.

#### Web Search

In [ ]:
models[1] = ('gpt-4o-search-preview', 'openai_chat')

In [ ]:
msg1 = mk_user_msg('What is the weather in Istanbul')
comps = {mn: await acomplete([msg1], model=mn, vendor_name=vnm, web_search_options={}) for mn,vnm in models}
for mn1,_ in models:
    for mn,vnm in models:
        msg2, msg3 = comps[mn1].message, mk_user_msg('Now check Brisbane')
        print(f'  {mn1:12} -> {mn:12}: ', end='')
        await print_stream([msg1, msg2, msg3], model=mn, vendor_name=vnm, web_search_options={})
        print()

  gpt-4o-mini  -> gpt-4o-mini : May 29, 2026, 11:51:27 AM

## Weather for Brisbane, Australia:
Current Conditions: Cloudy, 70°F (21°…


  gpt-4o-mini  -> gpt-4o-search-preview: As

 of

11

...

 is the current weather for **Brisbane, Queensland, Australia** on Friday

, May 29, 2026:

### 🌤️…


  models/gemini-3-flash-preview -> models/gemini-3-flash-preview: As of late morning on Friday, May 29, 2026, the weather in Brisbane is

 currently mostly sunny and pl…


  models/gemini-3-flash-preview -> models/gemini-3-flash-preview: In Brisbane, Australia, it is currently **late evening/early morning** (approximately 12:45

#### Codex

In [ ]:
cli, api_name, vendor_name = mk_client(model='gpt-5.4', vendor_name='codex')

In [ ]:
cli.ops[0].base_url

'https://chatgpt.com/backend-api/codex'

In [ ]:
d = Path('~/.codex/auth.json').expanduser().read_json()
d['tokens'].keys()

dict_keys(['id_token', 'access_token', 'refresh_token', 'account_id'])

In [ ]:
os.environ['CODEX_AUTH_TOKEN'] = nested_idx(Path('~/.codex/auth.json').expanduser().read_json(), 'tokens', 'access_token')

In [ ]:
msg1 = mk_user_msg('Hi! What are you capable of, tell concisely')
comp = await print_stream([msg1], model='gpt-5.4', vendor_name='codex', system='You are Codex based on GPT 5...')

I

 can

 help

 with

:



...

,

 and

 summar

izing

 …

OpenAI Responses doesn't emit reasoning summary text unless `summary: "auto"` (or `"detailed"`) is set

In [ ]:
msg1 = mk_user_msg('What is 1233123 * 456231?')
comp = await print_stream([msg1], model='gpt-5.4', vendor_name='codex', reasoning_effort='medium', system='You are Codex based on GPT 5...')

123

312

3

 ×

456

...

,

939

,

413

**

In [ ]:
msg1 = mk_user_msg('What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')
r = await acomplete([msg1], model='gpt-5.4', vendor_name='codex', tools=[sch], system='You are Codex based on GPT 5...', stream=True)
async for comp in r: pass
trmsg = mk_tool_res_msg(comp.tool_calls, ('8','30'))
comp = await print_stream([msg1, comp.message, trmsg], model='gpt-5.4', vendor_name='codex', tools=[sch], system='You are Codex based on GPT 5...')

3

 +

5

 =

8

10

 +

20

 =

30

Codex uses `{"type":"web_search"}` not `{"type":"web_search_preview"}`

In [ ]:
msg1 = mk_user_msg('What is the weather in Istanbul')
comp = await print_stream([msg1], web_search_options={"type":"web_search"},  model='gpt-5.4', vendor_name='codex', tools=[sch], system='You are Codex based on GPT 5...')

Current weather in Istanbul

, Türkiye: Mostly

 cloudy, 64

°F (18°C

).

Forecast:


- Friday, May

 29: High

 …

In [ ]:
comp

<div class="prose" markdown="1">

Current weather in Istanbul, Türkiye: Mostly cloudy, 64°F (18°C).

Forecast:
- Friday, May 29: High 73°F (23°C), Low 54°F (12°C), clouds early then sun
- Saturday, May 30: High 76°F (24°C), Low 64°F (18°C), brief morning shower possible
- Sunday, May 31: High 78°F (26°C), Low 62°F (17°C), mainly cloudy

If you want, I can also give it in °C only or provide a more detailed 7-day forecast.

🔧 web_search({'type': 'search', 'queries': ['weather: Istanbul, Turkey'], 'query': 'weather: Istanbul, Turkey'})


<details markdown='1'>

- model: `gpt-5.4`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=3121, completion_tokens=154, total_tokens=3275, cached_tokens=2432, cache_creation_tokens=0, reasoning_tokens=26, raw={'input_tokens': 3121, 'input_tokens_details': {'cached_tokens': 2432}, 'output_tokens': 154, 'output_tokens_details': {'reasoning_tokens': 26}, 'total_tokens': 3275})`

</details>

</div>

Reasoning Effort

In [ ]:
msg1 = mk_user_msg('What is 31231231 * 12312?')
comp = await print_stream([msg1], reasoning_effort={"effort": "low", "summary": "auto"},  model='gpt-5.4', vendor_name='codex', system='You are Codex based on GPT 5...')

🧠

🧠

🧠

🧠

🧠

...

384

518

916

072

**

In [ ]:
comp

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

**Calculating multiplication**

I need to compute the multiplication of 31,231,231 by 12,312. First, I break it down: 31,231,231 times 12,000 gives me 374,774,772,000. Then, for 312, I can split it into 300 and 12. So, 31,231,231 times 300 results in 9,369,369,300, while 31,231,231 times 12 is 374,774,772. When I sum it all up, I get 384,518,916,072.

</details>

31231231 × 12312 = **384518916072**

<details markdown='1'>

- model: `gpt-5.4`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=31, completion_tokens=155, total_tokens=186, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=135, raw={'input_tokens': 31, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 155, 'output_tokens_details': {'reasoning_tokens': 135}, 'total_tokens': 186})`

</details>

</div>

### Test Costs

In [ ]:
import litellm

#### OpenAI Chat

In [ ]:
# TODO: litellm pricing doesn't match https://app.fireworks.ai/models/fireworks/kimi-k2p6, ours do
# mn = 'accounts/fireworks/models/kimi-k2p5'
# comp = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model=mn, vendor_name='fireworks_ai', max_tokens=64, temperature=1.0)
# litecomp = litellm.completion(model=f"fireworks_ai/{mn}", messages=[{"role":"user","content":"Say hello in French"}], max_tokens=64, temperature=1.0)
# litellm_cost = litellm.completion_cost(completion_response=litecomp)
# test_close(litellm_cost, comp.cost, 1e-5)

In [ ]:
wav_data = httpx.get("https://openaiassets.blob.core.windows.net/$web/API/docs/audio/alloy.wav").content
audio_b64 = f"data:audio/wav;base64,{base64.b64encode(wav_data).decode()}"

In [ ]:
comp = await acomplete([Msg(role='user', content=[Part(type=PartType.input_audio, text=audio_b64), Part(type=PartType.text, text='What is this audio saying?')])], model='gpt-audio-1.5', vendor_name='openai_chat', temperature=1.0)
litecomp = litellm.completion(
    model='gpt-audio-1.5',
    messages=[{"role":"user","content":[
        {"type":"input_audio","input_audio":{"data":base64.b64encode(wav_data).decode(),"format":"wav"}},
        {"type":"text","text":"What is this audio saying?"}
    ]}],
    temperature=1.0
)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

#### OpenAI Responses

Text

In [ ]:
msg = mk_user_msg("Say hello in French")

In [ ]:
comp = await acomplete([msg], model='gpt-4o-mini', max_tokens=64, temperature=0.0)
litecomp = litellm.responses(model="gpt-4o-mini", input=[{"type":"message","role":"user","content":[{"type":"input_text","text":"Say hello in French"}]}], max_output_tokens=64, temperature=0.0)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([msg], model='gpt-4o-mini', max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# o-series models on Responses API use `max_output_tokens`, not `max_tokens`
comp = await acomplete([msg], model='o4-mini', temperature=1.0, max_tokens=64)
litecomp = litellm.responses(model="o4-mini", input=[{"type":"message","role":"user","content":[{"type":"input_text","text":"Say hello in French"}]}], temperature=1.0, max_output_tokens=64)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
it = await acomplete([msg], model='o4-mini', temperature=1.0, max_tokens=64, stream=True)
async for comp in it: pass
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

Image

In [ ]:
img_url = "https://img.freepik.com/free-photo/mountain-range-body-water_53876-139760.jpg?semt=ais_hybrid&w=740&q=80"
msg = Msg(role='user', content=[Part(type=PartType.input_image, text=img_url), Part(type=PartType.text, text='What is this image?')])
comp = await acomplete([msg], model='gpt-4o-mini', max_tokens=64, temperature=0.0)

In [ ]:
litecomp = litellm.responses(model="gpt-4o-mini",input=[{"type":"message","role":"user","content":[{"type":"input_image","image_url":img_url},{"type":"input_text","text":"What is this image?"}]}], max_output_tokens=64, temperature=0.0)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([msg], model='gpt-4o-mini', max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
# Comparing stream to non-stream, so use approx test
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-4)

#### Anthropic

Text

In [ ]:
msg = mk_user_msg("Say hello in French")

In [ ]:
comp = await acomplete([msg], model='claude-sonnet-4-20250514', max_tokens=64, temperature=0.0)
litecomp = litellm.completion(model="anthropic/claude-sonnet-4-20250514", messages=[{"role":"user","content":"Say hello in French"}], max_tokens=64, temperature=0.0)
test_eq(comp.cost, litellm.completion_cost(completion_response=litecomp))

In [ ]:
# Streaming - as a sanity check
it = await acomplete([msg], model='claude-sonnet-4-20250514', max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_eq(comp.cost, litellm.completion_cost(completion_response=litecomp))

Image

In [ ]:
msg = Msg('user', content=[Part(PartType.input_image, img_url), Part('text', 'What is this image?')])
comp = await acomplete([msg], model='claude-sonnet-4-20250514', max_tokens=64, temperature=0.0)
# litellm anthropic image form:
litecomp = litellm.completion(model="anthropic/claude-sonnet-4-20250514",
    messages=[{"role":"user","content":[{"type":"image_url","image_url":{"url":img_url}},{"type":"text","text":"What is this image?"}]}],
    max_tokens=64, temperature=0.0)
test_eq(comp.cost, litellm.completion_cost(completion_response=litecomp))

In [ ]:
# Streaming - as a sanity check
it = await acomplete([msg], model='claude-sonnet-4-20250514', max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_eq(comp.cost, litellm.completion_cost(completion_response=litecomp))

Cache read and write

In [ ]:
cc = {"cache_control": {"type": "ephemeral"}}
big_text = 'The quick brown fox jumps over the lazy dog. ' * 200
msg1 = Msg('user', content=[Part('text', big_text, data=cc), Part('text', 'Summarize')])
comp1 = await acomplete([msg1], model='claude-sonnet-4-20250514', max_tokens=64)  # writes cache
msg3 = Msg('user', content=[Part('text', 'Now in French')])
comp2 = await acomplete([msg1, comp1.message, msg3], model='claude-sonnet-4-20250514', max_tokens=64)  # reads cache

In [ ]:
big_msgs1 = [{"role":"user","content":[
    {"type":"text","text":big_text,"cache_control":{"type":"ephemeral"}},
    {"type":"text","text":"Summarize"}]}]
litecomp1 = litellm.completion(model="anthropic/claude-sonnet-4-20250514", messages=big_msgs1, max_tokens=64)

big_msgs2 = big_msgs1 + [
    {"role":"assistant","content":litecomp1.choices[0].message.content},
    {"role":"user","content":"Now in French"}]
litecomp2 = litellm.completion(model="anthropic/claude-sonnet-4-20250514", messages=big_msgs2, max_tokens=64)

We hit cache when we make the first api call with litellm. So testing only the second api call cost is enough:

In [ ]:
test_close(comp2.cost, litellm.completion_cost(completion_response=litecomp2))

In [ ]:
# Streaming - as a sanity check
it1 = await acomplete([msg1], model='claude-sonnet-4-20250514', max_tokens=64, stream=True)  # writes cache
it2 = await acomplete([msg1, comp1.message, msg3], model='claude-sonnet-4-20250514', max_tokens=64, stream=True)  # reads cache
async for comp1 in it1: pass
async for comp2 in it2: pass

#### Gemini

Basic call with flash/pro 3:

In [ ]:
flash_mn, pro_mn   = 'models/gemini-3-flash-preview', 'models/gemini-3.1-pro-preview'
lflash_mn, lpro_mn = 'gemini/gemini-3-flash-preview', 'gemini/gemini-3.1-pro-preview'

There is a slight difference in request payloads, fastllm uses the original request param field names from the spec which is camel case, and litellm normalizes them. Gemini api accepts both:

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model=flash_mn, max_tokens=64, temperature=0.0)
litecomp = litellm.completion(model=lflash_mn, messages=[{"role":"user","content":"Say hello in French"}], max_tokens=64, temperature=0.0)
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model=flash_mn, max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model=pro_mn, max_tokens=64, temperature=0.0)
litecomp = litellm.completion(model=lpro_mn, messages=[{"role":"user","content":"Say hello in French"}], max_tokens=64, temperature=0.0)
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-4)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Part('text', "Say hello in French")])], model=pro_mn, max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-4)

Thinking with flash/pro 3:

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', 'What is 123 * 456?')])], model=flash_mn, reasoning_effort='low', temperature=0.0)
litecomp = litellm.completion(model=lflash_mn, messages=[{"role":"user","content":"What is 123 * 456?"}], reasoning_effort='low', temperature=0.0)
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-4)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Part('text', 'What is 123 * 456?')])], model=flash_mn, reasoning_effort='low', temperature=0.0, stream=True)
async for comp in it: pass
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-3) # 0.02 cents difference due to 67 more thought tokens

In [ ]:
comp = await acomplete([Msg('user', content=[Part('text', 'What is 123 * 456?')])], model=pro_mn, reasoning_effort='low', temperature=0.0)
litecomp = litellm.completion(model=lpro_mn, messages=[{"role":"user","content":"What is 123 * 456?"}], reasoning_effort='low', temperature=0.0)
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-3)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([Msg('user', content=[Part('text', 'What is 123 * 456?')])], model=pro_mn, reasoning_effort='low', temperature=0.0, stream=True)
async for comp in it: pass
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-3)

Audio with flash/pro 3:

In [ ]:
msg = Msg('user', content=[Part(PartType.input_audio, audio_b64), Part('text', 'What is this audio saying?')])
comp = await acomplete([msg], model=flash_mn, max_tokens=64, temperature=0.0)
litecomp = litellm.completion(model=lflash_mn,
    messages=[{"role":"user","content":[{"type":"input_audio","input_audio":{"data":base64.b64encode(wav_data).decode(),"format":"wav"}},{"type":"text","text":"What is this audio saying?"}]}],
    max_tokens=64, temperature=0.0)
test_eq(litellm.completion_cost(completion_response=litecomp), comp.cost)

In [ ]:
# Streaming - as a sanity check
it = await acomplete([msg], model=flash_mn, max_tokens=64, temperature=0.0, stream=True)
async for comp in it: pass
test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-4)

JSON payload key differences (camel case vs snake in `inlineData`) result in no cache hit and different completions, so we use `test_close` here:

In [ ]:
msg = Msg('user', content=[Part(PartType.input_audio, audio_b64), Part('text', 'What is this audio saying?')])
comp = await acomplete([msg], model=pro_mn, temperature=0.0)
litecomp = litellm.completion(model=lpro_mn, messages=[{"role":"user","content":[{"type":"input_audio","input_audio":{"data":audio_b64.split(',', 1)[1],"format":"wav"}},{"type":"text","text":"What is this audio saying?"}]}], temperature=0.0)
# test_close(litellm.completion_cost(completion_response=litecomp), comp.cost, 1e-3)
litellm.completion_cost(completion_response=litecomp), comp.cost

(0.0033380000000000003, 0.006085999999999999)

In [ ]:
it = await acomplete([msg], model=pro_mn, temperature=0.0, stream=True)
async for comp in it: pass
litellm.completion_cost(completion_response=litecomp), comp.cost

(0.0033380000000000003, 0.00455)

The larger gap here isn't a costing bug — it's that the camelCase `inlineData` payload misses cachy, so each call hits the API fresh and Gemini Pro 3 generates a different response (more thought tokens + more output tokens). Cost functions are still correct given each response's `usage.raw`.

#### Caching with flash/pro 3:

In [ ]:
big_text = 'The quick brown fox jumps over the lazy dog. ' * 250
msg1 = Msg('user', content=[Part('text', big_text + "\n\nWrite sentence that is repeated")])
comp1 = await acomplete([msg1], model=flash_mn, temperature=0.0)

In [ ]:
msg3 = Msg('user', content=[Part('text', 'Now in French!')])
comp2 = await acomplete([msg1, comp1.message, msg3], model=flash_mn, temperature=0.0)  # should read
print('cached:', comp2.usage.raw.get('cachedContentTokenCount', 0))

cached: 2024


In [ ]:
big_text = 'The quick brown fox jumps over the lazy dog. ' * 250
big_msgs1 = [{"role":"user","content":big_text + "\n\nWrite sentence that is repeated"}]
litecomp1 = litellm.completion(model=lflash_mn, messages=big_msgs1, temperature=0.0)

In [ ]:
big_msgs2 = big_msgs1 + [{"role":"assistant","content":litecomp1.choices[0].message.content}, {"role":"user","content":"Now in French!"}]
litecomp2 = litellm.completion(model=lflash_mn, messages=big_msgs2, temperature=0.0)

Another difference between fastllm and litellm is the url construction, litellm uses `https://generativelanguage.googleapis.com/{v1alpha|v1beta}/models/{model}:generateContent?key={API_KEY}` and in fastllm we use spec's base_url which is `https://generativelanguage.googleapis.com/` and endpoints come directly from the spec, e.g. `v1beta/...`, also the api key is sent as a header

Gemini implicit caching is not guaranteed, so it is particularly difficult to test deterministically:

In [ ]:
litellm.completion_cost(completion_response=litecomp2), comp2.cost

(0.0015912, 0.0015912)

In [ ]:
comp2.usage.raw

<div class="prose" markdown="1">

```python
{ 'candidatesTokenCount': 28,
  'promptTokenCount': 2532,
  'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 2532}],
  'thoughtsTokenCount': 351,
  'totalTokenCount': 2911}
```

</div>

In [ ]:
litecomp2.usage

Usage(completion_tokens=412, prompt_tokens=2532, total_tokens=2944, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=384, rejected_prediction_tokens=None, text_tokens=28, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=2024, text_tokens=508, image_tokens=None, video_tokens=None), cache_read_input_tokens=2024)

#### Large context with flash/pro 3:

In [ ]:
from litellm.types.utils import ModelResponse, Usage as LUsage, PromptTokensDetailsWrapper, CompletionTokensDetailsWrapper

In [ ]:
from fastllm.gemini import cost as gemini_cost

In [ ]:
fake_usage = Usage(prompt_tokens=250_000, completion_tokens=1000, total_tokens=251_000,
                   raw={'promptTokenCount':250_000, 'candidatesTokenCount':1000, 'totalTokenCount':251_000})
pro_meta = dict2obj(get_model_meta('models/gemini-3-pro-preview', 'gemini'))
cost = gemini_cost(fake_usage, pro_meta)
expected = 250_000 * 4e-06 + 1000 * 1.8e-05  # above-200k rates
test_close(cost, expected, 1e-10)

In [ ]:
lusage = LUsage(
    prompt_tokens=250_000, completion_tokens=1000, total_tokens=251_000,
    prompt_tokens_details=PromptTokensDetailsWrapper(text_tokens=250_000),
    completion_tokens_details=CompletionTokensDetailsWrapper(text_tokens=1000, reasoning_tokens=0),
)
fake_resp = ModelResponse(model='gemini-3-pro-preview', usage=lusage)
fake_resp._hidden_params = {'custom_llm_provider': 'gemini'}

In [ ]:
test_eq(cost,litellm.completion_cost(completion_response=fake_resp))

### Debug Mode

In [ ]:
defaults.debug_mode = True

In [ ]:
tools = [sch]
msg1 = mk_user_msg('What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')
mn, vnm = models[0]
comps = await acomplete([msg1], model=mn, vendor_name=vnm, tools=tools)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fastllm debug  model=gpt-4o-mini vendor=openai api=openai base_url=https://api.openai.com/v1 path=/responses
────────────────────────────────────────────────────────────
payload:
{'model': 'gpt-4o-mini',
 'input': [{'type': 'message',
            'role': 'user',
            'content': [{'type': 'input_text',
                         'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}],
 'tools': [{'type': 'function',
            'name': 'simple_add',
            'description': 'add numbers',
            'parameters': {'type': 'object',
                           'properties': {'a': {'description': '', 'type': 'integer'},
                                          'b': {'description': '', 'type': 'integer'}},
                           'required': ['a', 'b']}}]}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [ ]:
tools = [sch]
msg1 = mk_user_msg('What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')
mn, vnm = models[2]
comps = await acomplete([msg1], model=mn, vendor_name=vnm, tools=tools, reasoning_effort='high')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fastllm debug  model=claude-sonnet-4-6 vendor=anthropic api=anthropic base_url=https://api.anthropic.com path=/v1/messages
────────────────────────────────────────────────────────────
payload:
{'model': 'claude-sonnet-4-6',
 'messages': [{'role': 'user',
               'content': [{'type': 'text',
                            'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}],
 'max_tokens': 1024,
 'tools': [{'name': 'simple_add',
            'description': 'add numbers',
            'input_schema': {'type': 'object',
                             'properties': {'a': {'description': '', 'type': 'integer'},
                                            'b': {'description': '', 'type': 'integer'}},
                             'required': ['a', 'b']}}],
 'thinking': {'type': 'adaptive'},
 'output_config': {'effort': 'high'}}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()